# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [ ]:
!pip install -U vllm

In [ ]:
!pip uninstall -y torchcodec sentence-transformers
!pip install "sentence-transformers==3.3.1"

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
#MAX_TOKENS  = 4096
MAX_TOKENS  = 4096

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 3.1 Create Embeddings and Cluster

In [6]:
# SBERT Embedding 

from sentence_transformers import SentenceTransformer

# Load model and define sentences
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = []

# Look at all questions and put them in texts
for item in data: 
    q = item["question"]
    options = item.get("options")

    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = " ".join(f"{lbl}. {opt}" for lbl, opt in zip(labels, options))
        text = f"Question: {q} Options: {opts_text}"
    else:
        text = f"Question: {q}"
    
    texts.append(text)

# Encode all at once (fast + efficient)
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(embeddings.shape)  
# (1126, 384) - 1126 questions with a 384 dimension vector

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

(943, 384)


In [7]:
# KMean Clustering
from sklearn.cluster import KMeans
import numpy as np

k = 8

kmeans = KMeans(n_clusters=k, random_state=23, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings)

print(cluster_ids.shape)       
print(kmeans.cluster_centers_.shape) 

clusters = {}
for cid in range(k):
    clusters[cid] = {
        "center": kmeans.cluster_centers_[cid],
        "points": []
    }

for idx, cid in enumerate(cluster_ids):
    clusters[cid]["points"].append(idx)

print("Number of clusters:", len(clusters))
for cid in clusters:
    print(f"Cluster {cid}: {len(clusters[cid]['points'])} questions")

(943,)
(8, 384)
Number of clusters: 8
Cluster 0: 74 questions
Cluster 1: 116 questions
Cluster 2: 138 questions
Cluster 3: 77 questions
Cluster 4: 169 questions
Cluster 5: 35 questions
Cluster 6: 190 questions
Cluster 7: 144 questions


In [8]:
import numpy as np

representatives_per_cluster = 2
cluster_rep_indices = {}
cluster_reps = {}

for cid in range(k):
    # indices of question in cluster
    cluster_idxs = np.where(cluster_ids == cid)[0]

    # embeddings of question in cluster
    cluster_embeds = embeddings[cluster_idxs]

    # centroid of the cluster
    centroid = kmeans.cluster_centers_[cid]

    # dist of each point to center
    dists = np.linalg.norm(cluster_embeds - centroid, axis=1)

    # pick 2 clistest point
    sorted_local = np.argsort(dists)[:representatives_per_cluster]

    # ma
    chosen_global = cluster_idxs[sorted_local]
    
    cluster_rep_indices[cid] = chosen_global.tolist()
    cluster_reps[cid] = [data[idx] for idx in chosen_global]

print(cluster_rep_indices)

{0: [253, 398], 1: [411, 401], 2: [886, 192], 3: [183, 394], 4: [314, 65], 5: [402, 266], 6: [309, 548], 7: [501, 523]}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [9]:
SYSTEM_PROMPT_MATH_REPRESENTATIVES = (
    "You are an expert mathematician. "
    "MANDATORY RULE: If multiple sub-answers are required, put them in one box seperated by commas e.g \\boxed{3,7} "
    "If this is not the first token, the response if invalid "
    "Do NOT think before answering and do NOT delat the boxed answer"
    "THEN give a concise explanation."
)

SYSTEM_PROMPT_MCQ_REPRESENTATIVES = (
    "You are an expert mathematician. "
    "MANDATORY RULE: FIRST line must be the SINGLE best answer inside \\boxed{}, e.g. \\boxed{C}. "
    "If this is not the first token, the response if invalid "
    "Do NOT think before answering and do NOT delat the boxed answer"
    "Do not write more than 120 words."
)

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "MANDATORY RULE: If multiple sub-answers are required, put them in one box seperated by commas e.g \\boxed{3,7} "
    "If this is not the first token, the response if invalid "
    "Do NOT think before answering and do NOT delat the boxed answer"
    "THEN give a concise explanation."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "MANDATORY RULE: FIRST line must be the SINGLE best answer inside \\boxed{}, e.g. \\boxed{C}. "
    "If this is not the first token, the response if invalid "
    "Do NOT think before answering and do NOT delat the boxed answer"
    "Do not write more than 120 words."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ_REPRESENTATIVES, f"{question}\n\nOptions:\n{opts_text}"

    return SYSTEM_PROMPT_MATH_REPRESENTATIVES, question

# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

def build_fewshot_prompt(item, exemplars):
    parts = []

    # Add exemplars
    for ex in exemplars:
        q = ex["question"]
        options = ex.get("options")
        sol = ex["solution"]

        if options:
            labels = [chr(65 + i) for i in range(len(options))]
            opts_text = "\n".join(
                f"{lbl}. {opt.strip()}"
                for lbl, opt in zip(labels, options)
            )

            block = (
                f"Example Problem:\n{q}\n\n"
                f"Options:\n{opts_text}\n\n"
                f"Step-by-step Solution:\n{sol}\n"
            )
        else:
            block = (
                f"Example Problem:\n{q}\n\n"
                f"Step-by-step Solution:\n{sol}\n"
            )

        parts.append(block)

    # Add target question
    q = item["question"]
    opts = item.get("options")

    if opts:
        labels = [chr(65 + i) for i in range(len(opts))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}"
            for lbl, opt in zip(labels, opts)
        )

        target = (
            "Now solve this new problem in the same style.\n"
            "At the end, include the final answer inside \\boxed{}.\n\n"
            f"Problem:\n{q}\n\n"
            f"Options:\n{opts_text}"
        )
    else:
        target = (
            "Now solve this new problem in the same style.\n"
            "At the end, include the final answer inside \\boxed{}.\n\n"
            f"Problem:\n{q}"
        )

    parts.append(target)

    user_prompt = "\n\n".join(parts)

    if item.get("options"):
        system_prompt = SYSTEM_PROMPT_MCQ
    else:
        system_prompt = SYSTEM_PROMPT_MATH

    return system_prompt, user_prompt

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [8]:
import gc

# Delete model and tokenizer if they exist
try:
    del model
    del tokenizer
except:
    pass

# Clear Python garbage
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()

# Reset cached memory stats
torch.cuda.reset_peak_memory_stats()

print("Memory cleared!")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Memory cleared!
Allocated: 0.01 GB
Cached: 0.02 GB


In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side='left',
)

In [11]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side="left",
)

llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on cuda.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [12]:
rep_prompts = []
rep_items = []

# Go through ALL representatives
for cid, rep_idxs in cluster_rep_indices.items():
    for idx in rep_idxs:
        item = data[idx]

        system, user = build_prompt(item["question"], item.get("options"))

        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )

        rep_prompts.append(prompt_text)
        rep_items.append(item)

print(f"Generating representative solutions for {len(rep_prompts)} examples...")

rep_outputs = generate_batch(
    rep_prompts,
    max_new_tokens=600,
    do_sample=False, # more stable reasoning
    batch_size=1
)

# Store them
def shorten_solution(text, max_chars=1200):
    if "\\boxed{" in text:
        box_pos = text.rfind("\\boxed{")
        return text[:box_pos + 200]
    return text[:max_chars]

rep_solutions = {}
for item, output in zip(rep_items, rep_outputs):
    rep_solutions[item["id"]] = shorten_solution(output)

print("Representative solutions generated!")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating representative solutions for 16 examples...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

Representative solutions generated!


In [ ]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=2048,
)

In [ ]:
def generate_batch_vllm(prompts, batch_size=16):
    all_responses = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]

        outputs = llm.generate(batch_prompts, sampling_params)

        for output in outputs:
            all_responses.append(output.outputs[0].text.strip())

        print(f"Generated {start + len(batch_prompts)} / {len(prompts)}")

    return all_responses

In [ ]:
rep_prompts = []
rep_items = []

for cid, rep_idxs in cluster_rep_indices.items():
    for idx in rep_idxs:
        item = data[idx]

        system, user = build_prompt(item["question"], item.get("options"))

        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )

        rep_prompts.append(prompt_text)
        rep_items.append(item)

print(f"Generating representative solutions for {len(rep_prompts)} examples...")

rep_outputs = generate_batch_vllm(
    rep_prompts,
    batch_size=8
)

rep_solutions = {}

for item, output in zip(rep_items, rep_outputs):
    rep_solutions[item["id"]] = output

print("Representative solutions generated!")

In [ ]:
bad = []

for k, v in rep_solutions.items():
    if len(v.strip()) == 0 or "\\boxed" not in v:
        bad.append(k)

print(f"{len(bad)} bad exemplars:", bad[:10])

In [ ]:
import os
import json
import pandas as pd

OUTPUT_CSV = "submission_partial.csv"
BATCH_SIZE = 100
START_INDEX = 0   # change this if you want, but skipping handles most reruns

# Load existing completed IDs
if os.path.exists(OUTPUT_CSV):
    existing_df = pd.read_csv(OUTPUT_CSV)
    completed_ids = set(existing_df["id"].astype(str))
    print(f"Found {len(completed_ids)} completed answers.")
else:
    completed_ids = set()

selected_items = data[:]

for item, cid in zip(data, cluster_ids):
    item["cluster_id"] = int(cid)

# Only keep unanswered items
remaining_items = [
    item for item in selected_items
    if str(item["id"]) not in completed_ids
]

# Take next batch
batch_items = remaining_items[START_INDEX : START_INDEX + BATCH_SIZE]

print(f"Running {len(batch_items)} questions this time.")

prompts = []

for item in batch_items:
    cid = item["cluster_id"]
    rep_idxs = cluster_rep_indices[cid]

    exemplars = []
    for idx in rep_idxs:
        ex_item = data[idx]

        if ex_item["id"] == item["id"]:
            continue

        exemplars.append({
            "question": ex_item["question"],
            "options": ex_item.get("options"),
            "solution": rep_solutions[ex_item["id"]]
        })

    system, user = build_fewshot_prompt(item, exemplars)

    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    prompts.append(prompt_text)

new_rows = []

for i, prompt in enumerate(prompts):
    item = batch_items[i]

    try:
        output = generate_batch(
            [prompt],
            max_new_tokens=700
        )[0]

        row = {
            "id": item["id"],
            "answer": output
        }

        new_rows.append(row)

        # append immediately after each answer
        pd.DataFrame([row]).to_csv(
            OUTPUT_CSV,
            mode="a",
            header=not os.path.exists(OUTPUT_CSV),
            index=False
        )

        print(f"Saved {i + 1}/{len(prompts)} | id={item['id']}")

    except Exception as e:
        print(f"Failed on id={item['id']}: {e}")
        break

Found 323 completed answers.
Running 100 questions this time.
Saved 1/100 | id=323
Saved 2/100 | id=324
Saved 3/100 | id=325
Saved 4/100 | id=326
Saved 5/100 | id=327
Saved 6/100 | id=328
Saved 7/100 | id=329
Saved 8/100 | id=330
Saved 9/100 | id=331
Saved 10/100 | id=332
Saved 11/100 | id=333
Saved 12/100 | id=334
Saved 13/100 | id=335
Saved 14/100 | id=336
Saved 15/100 | id=337
Saved 16/100 | id=338


In [15]:
data[0].keys()

dict_keys(['question', 'id', 'cluster_id'])

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [14]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/943 [00:00<?, ?it/s]


KeyError: 'answer'

## 8. Summary

Print accuracy broken down by question type.

In [55]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    7  (14.29%)
  Free-form  :    1 /   10  (10.00%)
  Overall    :    2 /   17  (11.76%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [23]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 2 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [17]:
pip install pandas

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
start = len(responses)  # 137

for i in range(start, len(prompts)):
    output = generate_batch([prompts[i]])[0]
    responses.append(output)

    with open("partial_results.jsonl", "a") as f:
        f.write(json.dumps({
            "id": selected_items[i]["id"],
            "response": output
        }) + "\n")

    if i % 10 == 0:
        print(f"Done {i}/{len(prompts)}")

NameError: name 'responses' is not defined

In [1]:
print(len(selected_items))
print(len(responses))

NameError: name 'selected_items' is not defined

In [18]:
import pandas as pd

submission = pd.DataFrame({
    "id": [item["id"] for item in selected_items],
    "answer": responses
})

print(submission.shape)
submission.head()

ValueError: All arrays must be of the same length